## Ejecute todos los códigos que se presentan y describa las tareas que se realizan, en todos los casos presente la salida que produce el código.

### a) Describa que hace el siguiente código:

In [ ]:
# import subprocess
import signal

# Define a custom exception for timeout
class TimeoutException(Exception):
    pass

# Define a handler that will be called when the alarm is triggered
def timeout_handler(signum, frame):
    raise TimeoutException("Code execution timed out!")

# Function to run a command with a timeout
def run_command_with_timeout(command, timeout):
    try:
        # Set the signal handler and an alarm
        signal.signal(signal.SIGALRM, timeout_handler)
        signal.alarm(timeout)

        # Execute the shell command
        process = subprocess.run(command, shell=True, check=True)

        # Cancel the alarm if the process completes in time
        signal.alarm(0)
        return process.returncode
    except TimeoutException:
        print("Command timed out!")
        return None
    except subprocess.CalledProcessError as e:
        print(f"Command failed with error: {e}")
        return None
    finally:
        # Disable the alarm
        signal.alarm(0)

# Commands to run
commands = [
    "mkdir -p /home/admin/data/db",
    "mongod --dbpath /home/admin/data/db --bind_ip 127.0.0.1 --fork --logpath /home/admin/data/db/mongodb.log",
    "mongosh"
]

# Run each command with a timeout
for cmd in commands:
    print(f"Running command: {cmd}")
    run_command_with_timeout(cmd, timeout=10)


### b) Describa que hace el siguiente código:

In [ ]:
import pymongo
import pandas as pd
from pymongo import MongoClient

# Initialize MongoDB client and create a local database and collection
client = MongoClient("mongodb://localhost:27017/")
db = client["sample_db"]
collection = db["products"]
client.server_info()

# Clear existing data in the collection if it exists
collection.delete_many({})

# Sample data to insert into the MongoDB collection
data = [
    {"product_id": 1, "name": "Laptop", "category": "Electronics", "price": 1200, "quantity": 10},
    {"product_id": 2, "name": "Smartphone", "category": "Electronics", "price": 800, "quantity": 25},
    {"product_id": 3, "name": "Tablet", "category": "Electronics", "price": 600, "quantity": 15},
    {"product_id": 4, "name": "Headphones", "category": "Accessories", "price": 150, "quantity": 50},
    {"product_id": 5, "name": "Smartwatch", "category": "Accessories", "price": 300, "quantity": 30},
    {"product_id": 6, "name": "Camera", "category": "Photography", "price": 1000, "quantity": 8},
    {"product_id": 7, "name": "Tripod", "category": "Photography", "price": 100, "quantity": 20},
    {"product_id": 8, "name": "Monitor", "category": "Electronics", "price": 400, "quantity": 12},
    {"product_id": 9, "name": "Keyboard", "category": "Accessories", "price": 50, "quantity": 100},
    {"product_id": 10, "name": "Mouse", "category": "Accessories", "price": 40, "quantity": 75}
]

# Insert sample data into the collection
collection.insert_many(data)
print("Sample data inserted successfully!")

### c) Describa que hace el siguiente código:

In [ ]:
# Function to convert MongoDB query results to DataFrame for display
def to_dataframe(cursor):
    return pd.DataFrame(list(cursor))

# Simple Query - Find all products in Electronics category
electronics_products = collection.find({"category": "Electronics"})
electronics_df = to_dataframe(electronics_products)
electronics_df

### d) Describa que hace el siguiente código:

In [ ]:

# Aggregation Pipeline Example
pipeline = [
    {"$match": {"category": "Accessories"}},
    {"$sort": {"price": -1}},
    {"$group": {"_id": "$category", "total_quantity": {"$sum": "$quantity"}, "average_price": {"$avg": "$price"}}},
    {"$limit": 2}
]

aggregation_result = collection.aggregate(pipeline)
aggregation_df = to_dataframe(aggregation_result)
aggregation_df


### e) Describa que hace el siguiente código:

In [ ]:
# Update the price of the product with product_id 1 (Laptop)
update_result = collection.update_one({"product_id": 1}, {"$set": {"price": 1100}})
print(f"Matched {update_result.matched_count} document(s) and modified {update_result.modified_count} document(s).")

# Fetch the updated document
updated_product = collection.find_one({"product_id": 1})
updated_product


### f) Describa que hace el siguiente código:

In [ ]:
# Delete products in the 'Photography' category
delete_result = collection.delete_many({"category": "Photography"})
print(f"Deleted {delete_result.deleted_count} document(s).")

# Verify remaining documents
remaining_products = collection.find()
to_dataframe(remaining_products)

### g) Describa que hace el siguiente código:

In [ ]:
# Create an index on the 'name' field to optimize search queries
index_result = collection.create_index("name")
print(f"Index created: {index_result}")

# List all indexes
indexes = collection.index_information()
indexes

### h) Describa que hace el siguiente código:

In [ ]:
# Add a text index on the 'name' field for text search
collection.create_index([("name", "text")])

# Use text search to find products containing the word 'Smart'
text_search_result = collection.find({"$text": {"$search": "Smart"}})
to_dataframe(text_search_result)

### i) Describa que hace el siguiente código:

In [ ]:

# Create another collection for orders
orders_collection = db["orders"]
orders_collection.delete_many({})  # Clear existing data

# Sample order data
orders_data = [
    {"order_id": 1, "product_id": 1, "quantity": 2},
    {"order_id": 2, "product_id": 2, "quantity": 1},
    {"order_id": 3, "product_id": 5, "quantity": 4}
]
orders_collection.insert_many(orders_data)

# Perform a $lookup to join products with orders
pipeline = [
    {
        "$lookup": {
            "from": "orders",
            "localField": "product_id",
            "foreignField": "product_id",
            "as": "order_info"
        }
    },
    {"$match": {"order_info": {"$ne": []}}}
]

join_result = collection.aggregate(pipeline)
to_dataframe(join_result)


### j) Describa que hace el siguiente código:

In [ ]:
# Use $project to include only 'name' and 'price' fields in the output
project_result = collection.aggregate([
    {"$project": {"_id": 0, "name": 1, "price": 1}}
])
to_dataframe(project_result)

### k) Describa que hace el siguiente código:

In [ ]:
# Use $unwind to flatten the 'order_info' array
unwind_result = collection.aggregate([
    {
        "$lookup": {
            "from": "orders",
            "localField": "product_id",
            "foreignField": "product_id",
            "as": "order_info"
        }
    },
    {"$unwind": "$order_info"}
])
to_dataframe(unwind_result)

### l) Describa que hace el siguiente código:

In [ ]:
# Find products where the name starts with 'S'
regex_result = collection.find({"name": {"$regex": "^S"}})
to_dataframe(regex_result)

### j) Vea el siguiente video https://www.youtube.com/watch?v=lWMemPN9t6Q y ejecute lo que se realiza en el mismo a partir de la ejecución de "mongod --version". En lugar de usar en el shell "mongo", usar "mongosh". Recuerde agregar un comentario explicativo a sus capturas de pantalla.